In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("spark dataframe")\
    .getOrCreate()


data = [
    (101,"ravi",24000),
    (102,"sita",25000),
    (103,"kavya",35000),
    (104,"prashanth",30000)
]

columns = ["id","name","salary"]

df = spark.createDataFrame(data,columns)

display(df)



id,name,salary
101,ravi,24000
102,sita,25000
103,kavya,35000
104,prashanth,30000


In [0]:
# from pyspark.sql import SparkSession, functions as F
# from pyspark.sql.window import window
# spark = SparkSession.builder \
#     .appName("spark dataframe")\
#     .getOrCreate()

In [0]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("spark dataframe").getOrCreate()

data = [
    (101, "ravi", 24000),
    (102, "sita", 25000),
    (103, "kavya", 35000),
    (104, "prashanth", 30000),
    (105, "anil", 25000),
    (106, "meena", 40000),
]
columns = ["id", "name", "salary"]
df = spark.createDataFrame(data, columns)

# Window ordered by salary (descending) - the basis for all ranking functions
window_by_salary = Window.orderBy(F.col("salary").desc())

df_rank = df.withColumn("row_number", F.row_number().over(window_by_salary)) \
    .withColumn("rank", F.rank().over(window_by_salary)) \
    .withColumn("dense_rank", F.dense_rank().over(window_by_salary)) \
    .withColumn("percent_rank", F.round(F.percent_rank().over(window_by_salary), 2)) \
    .withColumn("ntile_3", F.ntile(3).over(window_by_salary))

print("Ranking functions (ordered by salary desc):")
df_rank.show()

# Note: rank() skips ties (1,2,2,4), dense_rank() does not (1,2,2,3)
# ntile(3) splits rows into 3 roughly equal groups - useful for tercile analysis

Ranking functions (ordered by salary desc):


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---+---------+------+----------+----+----------+------------+-------+
| id|     name|salary|row_number|rank|dense_rank|percent_rank|ntile_3|
+---+---------+------+----------+----+----------+------------+-------+
|106|    meena| 40000|         1|   1|         1|         0.0|      1|
|103|    kavya| 35000|         2|   2|         2|         0.2|      1|
|104|prashanth| 30000|         3|   3|         3|         0.4|      2|
|102|     sita| 25000|         4|   4|         4|         0.6|      2|
|105|     anil| 25000|         5|   4|         4|         0.6|      3|
|101|     ravi| 24000|         6|   6|         5|         1.0|      3|
+---+---------+------+----------+----+----------+------------+-------+



In [0]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("spark dataframe").getOrCreate()

# Monthly sales data - typical use case for lag/lead
data = [
    ("2026-01", "east", 1000),
    ("2026-02", "east", 1200),
    ("2026-03", "east", 900),
    ("2026-01", "west", 1500),
    ("2026-02", "west", 1300),
    ("2026-03", "west", 1800),
]
columns = ["month", "region", "sales"]
df = spark.createDataFrame(data, columns)

# A PARTITIONED window: each region gets its own independent ordering
window_by_region = Window.partitionBy("region").orderBy("month")

df_lag = df.withColumn("prev_month_sales", F.lag("sales", 1).over(window_by_region)) \
    .withColumn("next_month_sales", F.lead("sales", 1).over(window_by_region)) \
    .withColumn("first_sales", F.first("sales").over(window_by_region)) \
    .withColumn("last_sales", F.last("sales").over(window_by_region)) \
    .withColumn("delta_vs_prev", F.col("sales") - F.lag("sales", 1).over(window_by_region))

print("lag/lead per region (note: first/last rows have nulls for lag/lead):")
df_lag.show()

# Handle the nulls at partition boundaries with a default value
df.withColumn("prev_sales", F.lag("sales", 1, 0).over(window_by_region)).show()

lag/lead per region (note: first/last rows have nulls for lag/lead):
+-------+------+-----+----------------+----------------+-----------+----------+-------------+
|  month|region|sales|prev_month_sales|next_month_sales|first_sales|last_sales|delta_vs_prev|
+-------+------+-----+----------------+----------------+-----------+----------+-------------+
|2026-01|  east| 1000|            NULL|            1200|       1000|      1000|         NULL|
|2026-02|  east| 1200|            1000|             900|       1000|      1200|          200|
|2026-03|  east|  900|            1200|            NULL|       1000|       900|         -300|
|2026-01|  west| 1500|            NULL|            1300|       1500|      1500|         NULL|
|2026-02|  west| 1300|            1500|            1800|       1500|      1300|         -200|
|2026-03|  west| 1800|            1300|            NULL|       1500|      1800|          500|
+-------+------+-----+----------------+----------------+-----------+----------+------

In [0]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("spark dataframe").getOrCreate()

data = [
    ("2026-01", 1000),
    ("2026-02", 1200),
    ("2026-03", 900),
    ("2026-04", 1500),
    ("2026-05", 1100),
    ("2026-06", 2000),
]
columns = ["month", "sales"]
df = spark.createDataFrame(data, columns)

# Window with a frame: unbounded start -> current row = running (cumulative) total
window_cumulative = Window.orderBy("month").rowsBetween(Window.unboundedPreceding, Window.currentRow)

# Moving window: last 3 rows (including current) = 3-month moving average
window_3m = Window.orderBy("month").rowsBetween(-2, Window.currentRow)

df_frames = df.withColumn("running_total", F.sum("sales").over(window_cumulative)) \
    .withColumn("running_count", F.count("sales").over(window_cumulative)) \
    .withColumn("moving_avg_3m", F.round(F.avg("sales").over(window_3m), 2)) \
    .withColumn("moving_sum_3m", F.sum("sales").over(window_3m))

print("Cumulative and moving window frames:")
df_frames.show()

# rangeBetween works on actual column values, not row positions:
# order by the month as a date (converted to days since epoch with unix_date)
# and include the current month plus up to 31 days before it
window_range = Window.orderBy(F.unix_date(F.to_date(F.concat(F.col("month"), F.lit("-01"))))) \
    .rangeBetween(-31, 0)
print("rangeBetween (current + previous month by value):")
df.withColumn("sum_range", F.sum("sales").over(window_range)).show()

Cumulative and moving window frames:


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------+-----+-------------+-------------+-------------+-------------+
|  month|sales|running_total|running_count|moving_avg_3m|moving_sum_3m|
+-------+-----+-------------+-------------+-------------+-------------+
|2026-01| 1000|         1000|            1|       1000.0|         1000|
|2026-02| 1200|         2200|            2|       1100.0|         2200|
|2026-03|  900|         3100|            3|      1033.33|         3100|
|2026-04| 1500|         4600|            4|       1200.0|         3600|
|2026-05| 1100|         5700|            5|      1166.67|         3500|
|2026-06| 2000|         7700|            6|      1533.33|         4600|
+-------+-----+-------------+-------------+-------------+-------------+

rangeBetween (current + previous month by value):
+-------+-----+---------+
|  month|sales|sum_range|
+-------+-----+---------+
|2026-01| 1000|     1000|
|2026-02| 1200|     2200|
|2026-03|  900|     2100|
|2026-04| 1500|     2400|
|2026-05| 1100|     2600|
|2026-06| 2000|     3

In [0]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("spark dataframe").getOrCreate()

# Employee data across departments - for top-N per group patterns
data = [
    (1, "ravi", "eng", 24000),
    (2, "sita", "eng", 30000),
    (3, "kavya", "eng", 35000),
    (4, "prashanth", "sales", 28000),
    (5, "anil", "sales", 22000),
    (6, "meena", "sales", 26000),
    (7, "latha", "hr", 20000),
    (8, "deepa", "hr", 21000),
]
columns = ["id", "name", "dept", "salary"]
df = spark.createDataFrame(data, columns)

# Top-2 earners PER DEPARTMENT: rank within each partition, then filter
window_dept = Window.partitionBy("dept").orderBy(F.col("salary").desc())

df_topn = df.withColumn("rn", F.row_number().over(window_dept)) \
    .withColumn("cume_dist", F.round(F.cume_dist().over(window_dept), 2))

print("Row numbers within each dept (top-N candidates):")
df_topn.show()

print("Top-2 earners per dept (filter rn <= 2):")
df_topn.filter(F.col("rn") <= 2).drop("rn", "cume_dist").show()

# Why row_number and not rank here? rank/ dense_rank would keep 3+ rows on ties.
# row_number guarantees exactly N rows per partition.

Row numbers within each dept (top-N candidates):
+---+---------+-----+------+---+---------+
| id|     name| dept|salary| rn|cume_dist|
+---+---------+-----+------+---+---------+
|  3|    kavya|  eng| 35000|  1|     0.33|
|  2|     sita|  eng| 30000|  2|     0.67|
|  1|     ravi|  eng| 24000|  3|      1.0|
|  8|    deepa|   hr| 21000|  1|      0.5|
|  7|    latha|   hr| 20000|  2|      1.0|
|  4|prashanth|sales| 28000|  1|     0.33|
|  6|    meena|sales| 26000|  2|     0.67|
|  5|     anil|sales| 22000|  3|      1.0|
+---+---------+-----+------+---+---------+

Top-2 earners per dept (filter rn <= 2):
+---+---------+-----+------+
| id|     name| dept|salary|
+---+---------+-----+------+
|  3|    kavya|  eng| 35000|
|  2|     sita|  eng| 30000|
|  8|    deepa|   hr| 21000|
|  7|    latha|   hr| 20000|
|  4|prashanth|sales| 28000|
|  6|    meena|sales| 26000|
+---+---------+-----+------+



In [0]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("spark dataframe").getOrCreate()

# Order events with duplicate (id, updated_at) pairs - like a slowly-arriving source table
data = [
    (101, "2026-01-05 10:00:00", "created", "ravi"),
    (101, "2026-02-10 09:30:00", "address_changed", "ravi"),
    (101, "2026-02-10 09:30:00", "address_changed", "ravi"),  # exact duplicate of the row above
    (102, "2026-01-08 11:15:00", "created", "sita"),
    (102, "2026-03-01 16:45:00", "phone_changed", "sita"),
    (102, "2026-03-01 16:45:00", "phone_changed", "sita"),
    (103, "2026-01-20 08:00:00", "created", "kavya"),
]
columns = ["order_id", "updated_at", "status", "customer"]
df = spark.createDataFrame(data, columns).withColumn("updated_at", F.to_timestamp("updated_at"))

print("Raw events (duplicates for 101 and 102):")
df.show(truncate=False)

# Classic dedupe pattern: number rows per key, newest first, then keep rn = 1
window_latest = Window.partitionBy("order_id").orderBy(F.col("updated_at").desc())

df_deduped = df.withColumn("rn", F.row_number().over(window_latest)) \
    .filter(F.col("rn") == 1) \
    .drop("rn")

print("Latest event per order_id (deduped, one row per key):")
df_deduped.show(truncate=False)

# Tie-breaker tip: if two rows share the same timestamp, add a deterministic
# secondary sort (e.g. status) so the result is stable across runs:
window_stable = Window.partitionBy("order_id").orderBy(F.col("updated_at").desc(), F.col("status").desc())
df.withColumn("rn", F.row_number().over(window_stable)).filter("rn = 1").drop("rn").show(truncate=False)

Raw events (duplicates for 101 and 102):
+--------+-------------------+---------------+--------+
|order_id|updated_at         |status         |customer|
+--------+-------------------+---------------+--------+
|101     |2026-01-05 10:00:00|created        |ravi    |
|101     |2026-02-10 09:30:00|address_changed|ravi    |
|101     |2026-02-10 09:30:00|address_changed|ravi    |
|102     |2026-01-08 11:15:00|created        |sita    |
|102     |2026-03-01 16:45:00|phone_changed  |sita    |
|102     |2026-03-01 16:45:00|phone_changed  |sita    |
|103     |2026-01-20 08:00:00|created        |kavya   |
+--------+-------------------+---------------+--------+

Latest event per order_id (deduped, one row per key):
+--------+-------------------+---------------+--------+
|order_id|updated_at         |status         |customer|
+--------+-------------------+---------------+--------+
|101     |2026-02-10 09:30:00|address_changed|ravi    |
|102     |2026-03-01 16:45:00|phone_changed  |sita    |
|103    

In [0]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("spark dataframe").getOrCreate()

data = [
    (1, "ravi", "eng", 24000),
    (2, "sita", "eng", 30000),
    (3, "kavya", "eng", 35000),
    (4, "prashanth", "sales", 28000),
    (5, "anil", "sales", 22000),
    (6, "meena", "hr", 26000),
]
columns = ["id", "name", "dept", "salary"]
df = spark.createDataFrame(data, columns)

# Aggregate functions (sum/avg/count) over a window with NO orderBy:
# the frame defaults to the ENTIRE partition - one value repeated per row.
# Note: no WARN here because partitionBy IS defined.
window_dept_total = Window.partitionBy("dept")

# Unbounded window (no partition) = grand total on every row
window_grand_total = Window.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

df_shares = df.withColumn("dept_total", F.sum("salary").over(window_dept_total)) \
    .withColumn("dept_avg", F.round(F.avg("salary").over(window_dept_total), 2)) \
    .withColumn("dept_count", F.count("*").over(window_dept_total)) \
    .withColumn("grand_total", F.sum("salary").over(window_grand_total)) \
    .withColumn("pct_of_dept", F.round(F.col("salary") / F.col("dept_total") * 100, 1)) \
    .withColumn("pct_of_company", F.round(F.col("salary") / F.col("grand_total") * 100, 1))

print("Aggregates over the whole partition (repeated on each row):")
df_shares.show()

# The same thing in SQL OVER() syntax - handy to know for Spark SQL / DBSQL:
spark.sql("""
    SELECT name, dept, salary,
           SUM(salary) OVER (PARTITION BY dept) AS dept_total,
           AVG(salary) OVER (PARTITION BY dept) AS dept_avg,
           SUM(salary) OVER () AS grand_total,
           ROUND(salary / SUM(salary) OVER (PARTITION BY dept) * 100, 1) AS pct_of_dept
    FROM VALUES
        (1, 'ravi', 'eng', 24000),
        (2, 'sita', 'eng', 30000),
        (3, 'kavya', 'eng', 35000),
        (4, 'prashanth', 'sales', 28000),
        (5, 'anil', 'sales', 22000),
        (6, 'meena', 'hr', 26000)
    AS t(id, name, dept, salary)
""").show()

Aggregates over the whole partition (repeated on each row):


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---+---------+-----+------+----------+--------+----------+-----------+-----------+--------------+
| id|     name| dept|salary|dept_total|dept_avg|dept_count|grand_total|pct_of_dept|pct_of_company|
+---+---------+-----+------+----------+--------+----------+-----------+-----------+--------------+
|  1|     ravi|  eng| 24000|     89000|29666.67|         3|     165000|       27.0|          14.5|
|  2|     sita|  eng| 30000|     89000|29666.67|         3|     165000|       33.7|          18.2|
|  3|    kavya|  eng| 35000|     89000|29666.67|         3|     165000|       39.3|          21.2|
|  6|    meena|   hr| 26000|     26000| 26000.0|         1|     165000|      100.0|          15.8|
|  4|prashanth|sales| 28000|     50000| 25000.0|         2|     165000|       56.0|          17.0|
|  5|     anil|sales| 22000|     50000| 25000.0|         2|     165000|       44.0|          13.3|
+---+---------+-----+------+----------+--------+----------+-----------+-----------+--------------+

+--------

In [0]:
from pyspark.sql import SparkSession, functions as F

# Sample data with nulls and duplicates
data = [
    (101, "ravi", 24000),
    (102, "sita", None),
    (103, None, 35000),
    (101, "ravi", 24000),
    (104, "prashanth", None),
    (None, "kavya", 30000)
]
columns = ["id", "name", "salary"]
df = spark.createDataFrame(data, columns)

print("Original DataFrame:")
df.show()

# 1) fillna() - replace nulls
# Fill all numeric columns with 0
print("fillna(0) - fill all nulls with 0:")
df.fillna(0).show()

# Fill a specific column with a value
print("fillna on 'name' column with 'Unknown':")
df.fillna({"name": "Unknown", "salary": 25000}).show()

# 2) dropna() - remove rows containing nulls
# Drop rows that have ANY null (default behavior)
print("dropna() - drop rows with any null:")
df.dropna().show()

# Drop rows only if ALL values are null
print("dropna(how='all') - drop rows where every column is null:")
df.dropna(how="all").show()

# Drop rows with nulls in specific columns only
print("dropna(subset=['salary']) - drop rows where salary is null:")
df.dropna(subset=["salary"]).show()

# Keep rows with at least 2 non-null values
print("dropna(thresh=2) - keep rows with at least 2 non-nulls:")
df.dropna(thresh=2).show()

# 3) dropDuplicates() - remove duplicate rows
# Remove fully duplicate rows
print("dropDuplicates() - remove exact duplicate rows:")
df.dropDuplicates().show()

# Remove duplicates based on specific columns (keeps first occurrence)
print("dropDuplicates(['id']) - dedupe by id column only:")
df.dropDuplicates(["id"]).show()

# Compare with distinct() - same as dropDuplicates() with no subset
df.distinct().show()

Original DataFrame:
+----+---------+------+
|  id|     name|salary|
+----+---------+------+
| 101|     ravi| 24000|
| 102|     sita|  NULL|
| 103|     NULL| 35000|
| 101|     ravi| 24000|
| 104|prashanth|  NULL|
|NULL|    kavya| 30000|
+----+---------+------+

fillna(0) - fill all nulls with 0:
+---+---------+------+
| id|     name|salary|
+---+---------+------+
|101|     ravi| 24000|
|102|     sita|     0|
|103|     NULL| 35000|
|101|     ravi| 24000|
|104|prashanth|     0|
|  0|    kavya| 30000|
+---+---------+------+

fillna on 'name' column with 'Unknown':
+----+---------+------+
|  id|     name|salary|
+----+---------+------+
| 101|     ravi| 24000|
| 102|     sita| 25000|
| 103|  Unknown| 35000|
| 101|     ravi| 24000|
| 104|prashanth| 25000|
|NULL|    kavya| 30000|
+----+---------+------+

dropna() - drop rows with any null:
+---+----+------+
| id|name|salary|
+---+----+------+
|101|ravi| 24000|
|101|ravi| 24000|
+---+----+------+

dropna(how='all') - drop rows where every colum